In [3]:
from pathlib import Path

import yaml
import json

REPO_ROOT = Path.cwd().parent
CONFIG_PATH = REPO_ROOT / "configs" / "config.yaml"

In [2]:
with open(CONFIG_PATH, 'r') as file:
    config = yaml.safe_load(file)

train_data_path = REPO_ROOT / config["data"]["train"]
test_data_path = REPO_ROOT / config["data"]["test"]

train_datasets = config["training_set"]
test_datasets = config["testing_set"]

train_image_paths = []
test_image_paths = []

for dataset in train_datasets:
    infrared_imgs = sorted((train_data_path / dataset / "infrared").glob("*.jpg"))
    visible_imgs = sorted((train_data_path / dataset / "visible").glob("*.jpg"))
    
    assert len(infrared_imgs) == len(visible_imgs), (
        f"Mismatch in {dataset}: "
        f"{len(infrared_imgs)} infrared vs {len(visible_imgs)} visible"
    )

    for infrared_image, visible_image in zip(infrared_imgs, visible_imgs):
        train_image_paths.append((infrared_image, visible_image))

for dataset in test_datasets:
    infrared_imgs = sorted((test_data_path / dataset / "infrared").glob("*.jpg"))
    visible_imgs = sorted((test_data_path / dataset / "visible").glob("*.jpg"))
    
    assert len(infrared_imgs) == len(visible_imgs), (
        f"Mismatch in {dataset}: "
        f"{len(infrared_imgs)} infrared vs {len(visible_imgs)} visible"
    )

    for infrared_image, visible_image in zip(infrared_imgs, visible_imgs):
        test_image_paths.append((infrared_image, visible_image))

In [4]:
MANIFEST_DIR = REPO_ROOT / "manifests"
MANIFEST_DIR.mkdir(exist_ok=True)

def save_manifest(image_pairs, output_path):
    with open(output_path, "w") as f:
        for infrared_path, visible_path in image_pairs:
            item = {
                "infrared": str(infrared_path.relative_to(REPO_ROOT)),
                "visible": str(visible_path.relative_to(REPO_ROOT)),
            }
            f.write(json.dumps(item) + "\n")

save_manifest(train_image_paths, MANIFEST_DIR / "lasher_train.jsonl")
save_manifest(test_image_paths, MANIFEST_DIR / "lasher_test.jsonl")